In [24]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict
load_dotenv()

True

In [25]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

def calculate_sr(state: BatsmanState):
    sr = state['runs']*100/state['balls']
    return {'sr':sr}

def calculate_bpb(state: BatsmanState):
    bpb = state['balls']/(state['fours']+state['sixes'])
    return {'bpb':bpb}

def calculate_boundary_percent(state:BatsmanState)->BatsmanState:
    boundary_percent = ((state['fours']*4 + state['sixes']*6)/state['runs'])*100
    return {'boundary_percent':boundary_percent}

def summary(state: BatsmanState):
    state['summary'] = f"""
    Strike Rate - {state['sr']}\n
    Balls per boundary - {state['bpb']}\n
    Boundary Percent - {state['boundary_percent']}"""

    return state

In [26]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()


In [27]:
initial_state = {'runs':100, 'balls':50, 'fours':6, 'sixes':4}
final_state = workflow.invoke(initial_state)

print(final_state['summary'])



    Strike Rate - 200.0

    Balls per boundary - 5.0

    Boundary Percent - 48.0
